# Lab 1

---

## Part I — Setting up the Environment

Import the required numerical, data manipulation, plotting, and Scikit-Learn tools.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

plt.rcParams["figure.figsize"] = (10, 6)
print("Environment configured successfully.")

from pathlib import Path

Environment configured successfully.


## Part II — Obtain and Load the Dataset

In [ ]:
# Load and recreate train/test
candidates = [
    Path("datasets/housing/housing.csv"),
    Path("datasets/housing.csv"),
    Path("housing.csv"),
]
housing_file = next((path for path in candidates if path.is_file()), None)

if housing_file is None:
    raise FileNotFoundError("Could not find housing.csv. Check the dataset location.")

housing = pd.read_csv(housing_file)


display(housing.head())
housing.info()
display(housing.describe())

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


## Part III — Exploring the Data

In [ ]:
housing.hist(bins=50, figsize=(12, 8))
plt.tight_layout()
plt.show()

## Part IV — Creating a Test Set

In [ ]:
train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)
print(f"Training set: {len(train_set)} rows | Test set: {len(test_set)} rows")

## Part V — Stratified Sampling

In [ ]:
housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0., 1.5, 3., 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index]
    strat_test_set = housing.loc[test_index]

# Compare sampling distributions
overall = housing["income_cat"].value_counts(normalize=True).sort_index()
strat = strat_test_set["income_cat"].value_counts(normalize=True).sort_index()
rand = pd.cut(
    test_set["median_income"],
    bins=[0., 1.5, 3., 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
).value_counts(normalize=True).sort_index()

comp_df = pd.DataFrame({"Overall": overall, "Stratified": strat, "Random": rand})
comp_df["Rand Error %"] = 100 * (comp_df["Random"] - comp_df["Overall"]) / comp_df["Overall"]
comp_df["Strat Error %"] = 100 * (comp_df["Stratified"] - comp_df["Overall"]) / comp_df["Overall"]
display(comp_df.round(3))

# Remove income_cat attribute
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

housing = strat_train_set.copy()

## Part VI — Geographical Visualization

In [ ]:
housing.plot(
    kind="scatter", x="longitude", y="latitude", alpha=0.4,
    s=housing["population"]/100, label="population",
    c="median_house_value", cmap=plt.get_cmap("jet"), colorbar=True,
    figsize=(10, 7)
)
plt.title("California Housing Prices and Population Density")
plt.show()

## Part VII — Correlation Analysis

In [ ]:
corr_matrix = housing.corr(numeric_only=True)
print(corr_matrix["median_house_value"].sort_values(ascending=False))

In [ ]:
housing.plot(kind="scatter", x="median_income", y="median_house_value", alpha=0.1)
plt.title("Median Income vs Median House Value")
plt.show()

## Part VIII — Feature Engineering

In [ ]:
housing["rooms_per_household"] = housing["total_rooms"] / housing["households"]
housing["bedrooms_per_room"] = housing["total_bedrooms"] / housing["total_rooms"]
housing["population_per_household"] = housing["population"] / housing["households"]

corr_matrix = housing.corr(numeric_only=True)
print(corr_matrix["median_house_value"].sort_values(ascending=False))

## Part IX — Handling Missing Values

In [ ]:
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

housing_num = housing.drop("ocean_proximity", axis=1)
imputer = SimpleImputer(strategy="median")
imputer.fit(housing_num)

X = imputer.transform(housing_num)
print("Imputer learned medians:\n", pd.Series(imputer.statistics_, index=housing_num.columns))

## Part X — Handling Categorical Attributes

In [ ]:
housing_cat = housing[["ocean_proximity"]]
cat_encoder = OneHotEncoder()
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)

print("Encoded categories:", cat_encoder.categories_)
print("One-hot matrix shape:", housing_cat_1hot.shape)

## Part XI — Feature Scaling

In [ ]:
scaler = StandardScaler()
housing_num_scaled = scaler.fit_transform(housing_num)
print("Scaled Mean:", housing_num_scaled.mean(axis=0).round(2))
print("Scaled Std Dev:", housing_num_scaled.std(axis=0).round(2))

## Part XII — Building a Preprocessing Pipeline

In [ ]:
num_attribs = list(housing_num)
cat_attribs = ["ocean_proximity"]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", OneHotEncoder(), cat_attribs),
])

housing_prepared = full_pipeline.fit_transform(housing)
print("Final prepared data shape:", housing_prepared.shape)